# Importing Necessary Libraries

In [35]:
import os
import numpy as np
import pandas as pd
import missingno
import seaborn as sns
import plotly.express as px 
import plotly.graph_objects as go
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import euclidean_distances
from scipy.spatial.distance import cdist
from scipy import stats
import spotipy as sp
from spotipy.oauth2 import SpotifyClientCredentials
from collections import defaultdict
import difflib

import warnings
warnings.filterwarnings("ignore")

# Importing Dataset & Cleaning

In [36]:
# Check current working directory and list files for debugging
print("Current working directory:", os.getcwd())
print("Files in current directory:", os.listdir())
print("Files in 'data' directory:", os.listdir('data') if os.path.exists('data') else "'data' directory does not exist")

# Load data if files exist
df = pd.read_csv('data/data.csv')
df_year = pd.read_csv('data/data_by_year.csv')
df_genre = pd.read_csv('data/data_by_genres.csv')

Current working directory: c:\Users\Bannu\Downloads\chinnu\Spotify-Recommendation-System\Spotify-Recommendation-System
Files in current directory: ['.cache', '4.2.0', 'assets', 'data', 'main.ipynb', 'main1.ipynb', 'README.md', 'recommendations.png']
Files in 'data' directory: ['data.csv', 'data.xlsx', 'data_by_artist.csv', 'data_by_genres.csv', 'data_by_year.csv', 'data_w_genres.csv']


In [37]:
# Remove outliers using z-score
df = df[(np.abs(stats.zscore(df.select_dtypes(include=['float64', 'int64']))) < 3).all(axis=1)]

## Describing the dataset

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8993 entries, 1 to 9998
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   valence           8993 non-null   float64
 1   year              8993 non-null   int64  
 2   acousticness      8993 non-null   float64
 3   artists           8993 non-null   object 
 4   danceability      8993 non-null   float64
 5   duration_ms       8993 non-null   int64  
 6   energy            8993 non-null   float64
 7   explicit          8993 non-null   int64  
 8   id                8993 non-null   object 
 9   instrumentalness  8993 non-null   float64
 10  key               8993 non-null   int64  
 11  liveness          8993 non-null   float64
 12  loudness          8993 non-null   float64
 13  mode              8993 non-null   int64  
 14  name              8993 non-null   object 
 15  popularity        8993 non-null   int64  
 16  release_date      8993 non-null   object 
 17  

In [39]:
df['year'] = pd.Categorical(df['year'])
df['key'] = pd.Categorical(df['key'])
df['mode'] = pd.Categorical(df['mode'])

In [40]:
round(df.describe(include='all'), 2)

,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
count,8993.00,8993.0,8993.00,8993,8993.00,8993.00,8993.00,8993.0,8993,8993.00,8993.0,8993.00,8993.00,8993.0,8993,8993.00,8993,8993.00,8993.00
unique,NaN,51.0,NaN,2223,NaN,NaN,NaN,NaN,8993,NaN,12.0,NaN,NaN,2.0,7747,NaN,900,NaN,NaN
top,NaN,1956.0,NaN,['Francisco Canaro'],NaN,NaN,NaN,NaN,3mDOHVY9MWZOTODSNF5GBj,NaN,0.0,NaN,NaN,1.0,White Christmas,NaN,1933,NaN,NaN
freq,NaN,198.0,NaN,207,NaN,NaN,NaN,NaN,1,NaN,1189.0,NaN,NaN,6681.0,16,NaN,196,NaN,NaN
mean,0.55,NaN,0.80,NaN,0.53,192794.07,0.31,0.0,NaN,0.23,NaN,0.19,-13.57,NaN,NaN,24.46,NaN,0.07,113.53
std,0.26,NaN,0.25,NaN,0.16,69719.95,0.20,0.0,NaN,0.36,NaN,0.12,4.57,NaN,NaN,21.64,NaN,0.07,30.76
min,0.01,NaN,0.00,NaN,0.06,29832.00,0.00,0.0,NaN,0.00,NaN,0.02,-28.84,NaN,NaN,0.00,NaN,0.02,34.76
25%,0.34,NaN,0.74,NaN,0.41,157707.00,0.16,0.0,NaN,0.00,NaN,0.10,-16.08,NaN,NaN,4.00,NaN,0.04,89.67
50%,0.58,NaN,0.91,NaN,0.54,178867.00,0.28,0.0,NaN,0.00,NaN,0.14,-12.92,NaN,NaN,18.00,NaN,0.04,110.97
75%,0.77,NaN,0.98,NaN,0.65,201560.00,0.43,0.0,NaN,0.47,NaN,0.24,-10.36,NaN,NaN,44.00,NaN,0.06,130.59


In [41]:
df_genre.info() #checking datatype of columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2973 entries, 0 to 2972
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mode              2973 non-null   int64  
 1   genres            2973 non-null   object 
 2   acousticness      2973 non-null   float64
 3   danceability      2973 non-null   float64
 4   duration_ms       2973 non-null   float64
 5   energy            2973 non-null   float64
 6   instrumentalness  2973 non-null   float64
 7   liveness          2973 non-null   float64
 8   loudness          2973 non-null   float64
 9   speechiness       2973 non-null   float64
 10  tempo             2973 non-null   float64
 11  valence           2973 non-null   float64
 12  popularity        2973 non-null   float64
 13  key               2973 non-null   int64  
dtypes: float64(11), int64(2), object(1)
memory usage: 325.3+ KB


In [42]:
#converting numeric to categorical
df_genre['genres'] = pd.Categorical(df_genre['genres'])
df_genre['key'] = pd.Categorical(df_genre['key'])
df_genre['mode'] = pd.Categorical(df_genre['mode'])

In [43]:
round(df_genre.describe(include='all'), 2)

,mode,genres,acousticness,danceability,duration_ms,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,popularity,key
count,2973.0,2973,2973.00,2973.00,2973.00,2973.00,2973.00,2973.00,2973.00,2973.00,2973.00,2973.00,2973.00,2973.0
unique,2.0,2973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0
top,1.0,zydeco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0
freq,2477.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,694.0
mean,NaN,NaN,0.40,0.54,251720.85,0.56,0.21,0.19,-10.51,0.08,119.02,0.49,39.92,NaN
std,NaN,NaN,0.32,0.15,94656.86,0.23,0.27,0.09,5.37,0.08,17.47,0.20,16.75,NaN
min,NaN,NaN,0.00,0.06,30946.00,0.00,0.00,0.02,-41.82,0.02,47.14,0.00,0.00,NaN
25%,NaN,NaN,0.12,0.44,206378.85,0.40,0.00,0.14,-12.43,0.04,109.20,0.35,32.49,NaN
50%,NaN,NaN,0.32,0.55,237545.34,0.60,0.08,0.18,-9.22,0.06,119.19,0.50,43.06,NaN
75%,NaN,NaN,0.67,0.65,277272.00,0.73,0.34,0.22,-6.92,0.09,127.51,0.64,51.14,NaN


# Recommendation System

In [44]:
# Spotify API credentials
%env SPOTIFY_CLIENT_ID=cef661d1f6cb40c198af14b5c88db48c
%env SPOTIFY_CLIENT_SECRET=39328504d30a4a2f9b641a45463c58d1

# Initialize Spotify client
sp_call = sp.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.environ["SPOTIFY_CLIENT_ID"],
    client_secret=os.environ["SPOTIFY_CLIENT_SECRET"]
))

env: SPOTIFY_CLIENT_ID=cef661d1f6cb40c198af14b5c88db48c
env: SPOTIFY_CLIENT_SECRET=39328504d30a4a2f9b641a45463c58d1


In [45]:
def find_song(name, year):
    song_data = defaultdict()
    results = sp_call.search(q='track: {} year: {}'.format(name, year), limit=1)
    if results['tracks']['items'] == []:
        return None
    results = results['tracks']['items'][0]
    track_id = results['id']
    audio_features = sp_call.audio_features(track_id)[0]
    song_data['name'] = [name]
    song_data['year'] = [year]
    song_data['explicit'] = [int(results['explicit'])]
    song_data['duration_ms'] = [results['duration_ms']]
    song_data['popularity'] = [results['popularity']]
    for key, value in audio_features.items():
        song_data[key] = [value]
    return pd.DataFrame(song_data)

In [46]:
def get_song_data(song, spotify_data):
    try:
        song_data = spotify_data[(spotify_data['name'] == song['name']) 
                                & (spotify_data['year'] == song['year'])].iloc[0]
        return song_data
    except IndexError:
        print(f"Warning: {song['name']} ({song['year']}) not found in local dataset. Attempting Spotify API.")
        return find_song(song['name'], song['year'])

In [47]:
def get_mean_vector(song_list, spotify_data):
    song_vectors = []
    for song in song_list:
        song_data = get_song_data(song, spotify_data)
        if song_data is None:
            print(f"Warning: {song['name']} does not exist in Spotify or in database")
            continue
        try:
            song_vector = song_data[number_cols].values.astype(float)
            song_vectors.append(song_vector)
        except KeyError as e:
            print(f"Error: Missing feature {e} for song {song['name']} ({song['year']})")
            continue
    if not song_vectors:
        return None
    song_matrix = np.array(song_vectors)
    return np.mean(song_matrix, axis=0)

In [48]:
number_cols = [
    'valence', 'acousticness', 'danceability', 'duration_ms', 'energy', 
    'explicit', 'instrumentalness', 'liveness', 'loudness', 'popularity', 
    'speechiness', 'tempo'
]

def unnest_dictionary(song_list_dict):
    flattened_dict = defaultdict()
    for key in song_list_dict[0].keys():
        flattened_dict[key] = []
    for dictionary in song_list_dict:
        for key, value in dictionary.items():
            flattened_dict[key].append(value)
    return flattened_dict

def recommend_songs(song_list, spotify_data, n_songs=10):
    metadata_cols = ['name', 'year', 'artists']
    song_dict = unnest_dictionary(song_list)
    song_center = get_mean_vector(song_list, spotify_data)
    if song_center is None or np.any(np.isnan(song_center)):
        raise ValueError("Mean vector could not be computed. Please check your song list and data.")
    scaler = song_cluster_pipeline.named_steps['scaler']
    scaled_data = scaler.transform(spotify_data[number_cols])
    scaled_song_center = scaler.transform(song_center.reshape(1, -1))
    distances = cdist(scaled_song_center, scaled_data, 'cosine')
    index = list(np.argsort(distances)[:, :n_songs][0])
    recommended_songs = spotify_data.iloc[index]
    recommended_songs = recommended_songs[~recommended_songs['name'].isin(song_dict['name'])]
    return recommended_songs[metadata_cols].to_dict(orient='records')

def print_recommendations(recommendations):
    for i, song in enumerate(recommendations):
        print(f'Recommendation {i+1}:')
        print(f'Song Name: {song["name"]}')
        print(f'Year: {song["year"]}')
        print(f'Artists: {song["artists"]}')
        print()

def visualize_recommendations(recommendations):
    names = [song['name'] for song in recommendations]
    years = [song['year'] for song in recommendations]
    plt.figure(figsize=(10, 6))
    plt.barh(names, years, color='skyblue')
    plt.xlabel('Year')
    plt.title('Recommended Songs by Year')
    plt.gca().invert_yaxis()
    plt.savefig('recommendations.png')
    plt.close()

In [49]:
def find_song(name, year):
    song_data = defaultdict()
    try:
        results = sp_call.search(q='track: {} year: {}'.format(name, year), limit=1)
        if results['tracks']['items'] == []:
            return None
        results = results['tracks']['items'][0]
        track_id = results['id']
        audio_features = sp_call.audio_features(track_id)
        if not audio_features or audio_features[0] is None:
            print(f"Warning: No audio features found for {name} ({year})")
            return None
        audio_features = audio_features[0]
        song_data['name'] = [name]
        song_data['year'] = [year]
        song_data['explicit'] = [int(results['explicit'])]
        song_data['duration_ms'] = [results['duration_ms']]
        song_data['popularity'] = [results['popularity']]
        for key, value in audio_features.items():
            song_data[key] = [value]
        return pd.DataFrame(song_data)
    except sp.SpotifyException as e:
        print(f"Spotify API error for '{name}' ({year}): {e}")
        return None
    except Exception as e:
        print(f"Error retrieving song '{name}' ({year}): {e}")
        return None

In [50]:
# Define the clustering pipeline
song_cluster_pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('kmeans', KMeans(n_clusters=20, random_state=42))
])

# Sample input song list (using songs from the dataset)
song_list = [
    {'name': 'White Christmas', 'year': 1942},
    {'name': 'Silent Night', 'year': 1935},
    {'name': 'Jingle Bells', 'year': 1946}
]

# Fit the pipeline
song_cluster_pipeline.fit(df[number_cols])

# Generate and display recommendations
try:
    recommendations = recommend_songs(song_list, df, n_songs=10)
    print_recommendations(recommendations)
    visualize_recommendations(recommendations)
except ValueError as e:
    print(f"Error: {e}")


HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=4gq4rq9K5zN1wKwLjo8e2y with Params: {} returned 403 due to None


Spotify API error for 'Silent Night' (1935): http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=4gq4rq9K5zN1wKwLjo8e2y:
 None, reason: None


HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=5sAxRoAmdnxftcCq7gIjkC with Params: {} returned 403 due to None


Spotify API error for 'Jingle Bells' (1946): http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=5sAxRoAmdnxftcCq7gIjkC:
 None, reason: None
Recommendation 1:
Song Name: Long, Long, Long - Remastered 2009
Year: 1968
Artists: ['The Beatles']

Recommendation 2:
Song Name: El Condor Pasa (If I Could)
Year: 1970
Artists: ['Simon & Garfunkel']

Recommendation 3:
Song Name: Beyond the Sea (La Mer)
Year: 1962
Artists: ['Ray Conniff']

Recommendation 4:
Song Name: O Come All Ye Faithful
Year: 1962
Artists: ['Nat King Cole']

Recommendation 5:
Song Name: Change Partners
Year: 1967
Artists: ['Antônio Carlos Jobim', 'Frank Sinatra']

Recommendation 6:
Song Name: Have Yourself A Merry Little Christmas - Remastered
Year: 1957
Artists: ['Frank Sinatra']

Recommendation 7:
Song Name: In A Sentimental Mood
Year: 1963
Artists: ['Duke Ellington', 'John Coltrane']

Recommendation 8:
Song Name: Scarborough Fair / Canticle
Year: 1966
Artists: ['Simon & Garfunkel']

Recommendation 9:

# END